# 🌊 Huấn Luyện RT-DETR Nhận Diện Ngập Lụt trên Kaggle (1 Class: ngap_lut)

Notebook này hướng dẫn quy trình huấn luyện mô hình **RT-DETR (Ultralytics)** trên nền tảng **Kaggle** miễn phí với GPU T4 x2 hoặc P100.

### 🎯 Mục tiêu (Phương án 1):
- Huấn luyện phát hiện ngập lụt nhị phân: **Có ngập lụt (`0: ngap_lut`)** hoặc **Không ngập lụt**.
- Tự động đồng bộ đường dẫn dataset và gộp toàn bộ nhãn về Class 0.

### ⚠️ Lưu ý trước khi chạy:
1. Bật GPU trong Kaggle: **Settings** -> **Accelerator** -> Chọn **GPU T4 x2** hoặc **GPU P100**.
2. Bật Internet: **Settings** -> **Internet** -> Chọn **Internet on**.

## Bước 1: Cài đặt thư viện cần thiết

In [ ]:
!pip install -q ultralytics onnx onnxslim

## Bước 2: Kiểm tra GPU

In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")

## Bước 3: Tự động phát hiện Dataset & Tạo cấu hình `data_kaggle.yaml`

In [ ]:
import os
import glob

# Tự động tìm đường dẫn chứa dataset trong /kaggle/input/
found_train_imgs = glob.glob("/kaggle/input/**/images/train", recursive=True)
if found_train_imgs:
    dataset_dir = os.path.dirname(os.path.dirname(found_train_imgs[0]))
    print(f"✅ Đã tìm thấy dataset tại: {dataset_dir}")
else:
    dataset_dir = "/kaggle/input/flood-dataset"
    print(f"⚠️ Không tự động tìm thấy, sử dụng đường dẫn mặc định: {dataset_dir}")

yaml_content = f"""
path: {dataset_dir}
train: images/train
val: images/val

names:
  0: ngap_lut
"""

with open("/kaggle/working/data_kaggle.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content.strip())

print("✅ Đã tạo file cấu hình: /kaggle/working/data_kaggle.yaml")

## Bước 4: Khởi chạy Huấn luyện (Training RT-DETR-L)

In [ ]:
from ultralytics import RTDETR

# Khởi tạo pre-trained RT-DETR-L
model = RTDETR("rtdetr-l.pt")

# Huấn luyện mô hình tối ưu cho GPU T4/P100 (batch=8 chống OOM, amp=True tăng tốc)
results = model.train(
    data="/kaggle/working/data_kaggle.yaml",
    epochs=80,
    imgsz=640,
    batch=8,          # batch=8 tối ưu và an toàn tuyệt đối trên Kaggle GPU
    device=0,
    workers=4,
    amp=True,          # FP16 Automatic Mixed Precision
    patience=15,       # Early stopping sau 15 epochs không cải thiện
    project="/kaggle/working/runs_flood",
    name="rtdetr_kaggle_exp",
    exist_ok=True,
    save=True,
    plots=True
)

print("✅ Huấn luyện hoàn tất! Trọng số lưu tại: /kaggle/working/runs_flood/rtdetr_kaggle_exp/weights/best.pt")

## Bước 5: Đánh giá chất lượng mô hình (Validation)

In [ ]:
best_pt = "/kaggle/working/runs_flood/rtdetr_kaggle_exp/weights/best.pt"
eval_model = RTDETR(best_pt)

metrics = eval_model.val(data="/kaggle/working/data_kaggle.yaml", split="val")
print(f"🎯 mAP@50    : {metrics.box.map50 * 100:.2f}%")
print(f"🎯 mAP@50-95 : {metrics.box.map * 100:.2f}%")

## Bước 6: Export sang định dạng ONNX

In [ ]:
import os
import shutil

# Export ONNX
onnx_path = eval_model.export(
    format="onnx",
    imgsz=640,
    simplify=True,
    opset=16,
    batch=1
)

# Đổi tên và đưa ra thư mục /kaggle/working/ để tải về
final_onnx = "/kaggle/working/rt_detr_flood.onnx"
shutil.copy2(onnx_path, final_onnx)

print(f"🎉 Hoàn tất! File ONNX đã sẵn sàng tại: {final_onnx}")
print("👉 Bạn chỉ cần vào tab Output bên phải màn hình Kaggle và click Download file 'rt_detr_flood.onnx'.")